In [1]:
from specutils import Spectrum1D
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import numpy as np
import pandas as pd
import matplotlib.ticker as plticker
from astropy.stats import sigma_clip
from astropy.time import Time
import requests
import re
from io import BytesIO
from astropy import units as u
from pyvo.dal.ssa  import search, SSAService
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from PyAstronomy import pyasl
from tqdm import trange

from joblib import Parallel, delayed
from tqdm.notebook import tqdm

In [2]:
galah_table = Table.read("./galah_dr4_allstar_240705.fits")
galah_table_good_select_teff = galah_table['teff']/galah_table['e_teff']>10
galah_table_good_select_logg = galah_table['logg']/galah_table['e_logg']>10
galah_table_good_select_flagsp = galah_table['flag_sp']==0
galah_table_good_select_flagfe = galah_table['flag_ba_fe']==0
galah_table_good_select = galah_table_good_select_teff & galah_table_good_select_logg & galah_table_good_select_flagsp & galah_table_good_select_flagfe
galah_table_good = galah_table[galah_table_good_select]

In [6]:
select = (galah_table_good['teff']<5100) & (galah_table_good['teff']>4700) & (galah_table_good['logg']>3.2) & \
         (galah_table_good['logg']<3.6) &\
         (galah_table_good['snr_px_ccd1']>75) & (galah_table_good['snr_px_ccd2']>75) &\
         (galah_table_good['snr_px_ccd3']>75) & (galah_table_good['snr_px_ccd4']>75)


23


In [88]:
url = (
    "https://datacentral.org.au/vo/slink/links"
    "?ID=200810001101299&DR=galah_dr4&IDX=1&FILT=B"
    "&DPSUBTYPE=combined&RESPONSEFORMAT=fits"
).format(sobject_id, f)

r = requests.get(url, timeout=60)
r.raise_for_status()

spec = Spectrum1D.read(
    BytesIO(r.content),
    format="wcs1d-fits"
)

primary_hdu = fits.PrimaryHDU(
                data=np.asarray(spec.flux.value)
            )

hdr = primary_hdu.header
for col in galah_table_good_select.colnames:
    key = col.upper()[:8]
    val = row[col]
    hdr[key] = val if np.isscalar(val) else str(val)

wave_hdu = fits.ImageHDU(
    data=np.asarray(spec.wavelength.value),
    name="WAVELENGTH"
)

fits.HDUList([primary_hdu, wave_hdu]).writeto(
    f"./galah_spectra_script/{sobject_id}{fc}.fits",
    overwrite=True
)

OSError: No SIMPLE card found, this file does not appear to be a valid FITS file. If this is really a FITS file, try with ignore_missing_simple=True